In [ ]:
import json
import itertools
from torchvision import models

# Confirm device before training
print(f'Training on: {device}')

# Define hyperparameter options to test
hyperparams = {
    'lr':         [1e-3, 5e-4, 1e-4],
    'optimizer':  ['SGD', 'AdamW', 'Adam'],
    'frozen':     [True, False],
    'batch_size': [32, 64],
}

# Store results for comparison
results = []

# Load previously saved results to allow resuming interrupted sweeps
try:
    with open('hyperparameter_results.json', 'r') as f:
        results = json.load(f)
    print(f'Loaded {len(results)} previously saved results')
except FileNotFoundError:
    print('No saved results found, starting fresh')

# Generate all combinations including batch size
combinations = list(itertools.product(
    hyperparams['lr'],
    hyperparams['optimizer'],
    hyperparams['frozen'],
    hyperparams['batch_size']
))

# Track already completed combinations to allow resuming
completed = [(r['optimizer'], r['lr'], r['frozen'], r['batch_size']) for r in results]

for lr, opt_name, frozen, batch_size in combinations:

    # Skip already completed combinations
    if (opt_name, lr, frozen, batch_size) in completed:
        print(f'Skipping: {opt_name} | lr={lr} | frozen={frozen} | batch={batch_size}')
        continue

    print(f'\n{"="*60}')
    print(f'Running: optimizer={opt_name} | lr={lr} | frozen={frozen} | batch_size={batch_size}')
    print(f'{"="*60}')

    # Recreate DataLoaders for each batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=6, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,num_workers=6, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,num_workers=6, pin_memory=True)
    # Re-initialize fresh model for each experiment
    model = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)

    # Freeze or unfreeze layers
    if frozen:
        for param in model.parameters():
            param.requires_grad = False
        model.fc.requires_grad_(True)
        params = model.fc.parameters()
    else:
        for param in model.parameters():
            param.requires_grad = True
        params = model.parameters()

    # Initialize optimizer
    if opt_name == 'SGD':
        optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9)
    elif opt_name == 'AdamW':
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    elif opt_name == 'Adam':
        optimizer = torch.optim.Adam(params, lr=lr)

    criterion = torch.nn.CrossEntropyLoss()

    # Train
    history = train(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)

    # Evaluate on test set
    model.eval()
    test_corrects = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            test_corrects += torch.sum(preds == labels.data)
    test_acc = test_corrects.float() / len(test_dataset)

    # Store result including batch size
    results.append({
        'optimizer':  opt_name,
        'lr':         lr,
        'frozen':     frozen,
        'batch_size': batch_size,
        'train_acc':  history['train_acc'][-1],
        'val_acc':    history['val_acc'][-1],
        'test_acc':   test_acc.item(),
    })

    # Save results after every combination in case of session reset
    with open('hyperparameter_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Results saved ({len(results)}/{len(combinations)} combinations done)')

# Print comparison table
print(f'\n{"="*80}')
print(f'{"Optimizer":<10} {"LR":<8} {"Frozen":<8} {"Batch":<8} {"Train Acc":<12} {"Val Acc":<12} {"Test Acc":<10}')
print(f'{"="*80}')
for r in results:
    print(f'{r["optimizer"]:<10} {r["lr"]:<8} {str(r["frozen"]):<8} {r["batch_size"]:<8} '
          f'{r["train_acc"]:<12.4f} {r["val_acc"]:<12.4f} {r["test_acc"]:<10.4f}')

# Pick best combination based on val accuracy
best = max(results, key=lambda x: x['val_acc'])
print(f'\nBest combination:')
print(f'  Optimizer  : {best["optimizer"]}')
print(f'  LR         : {best["lr"]}')
print(f'  Frozen     : {best["frozen"]}')
print(f'  Batch Size : {best["batch_size"]}')
print(f'  Val Acc    : {best["val_acc"]:.4f}')
print(f'  Test Acc   : {best["test_acc"]:.4f}')

Streaming output truncated to the last 5000 lines.
  Epoch [8/10] | Batch [80/102] | Loss: 0.0307 | Acc: 0.9875
  Epoch [8/10] | Batch [90/102] | Loss: 0.0012 | Acc: 0.9878
  Epoch [8/10] | Batch [100/102] | Loss: 0.0081 | Acc: 0.9887

--- Epoch [8/10] Summary ---
  Train Loss : 0.0305 | Train Acc : 0.9890
  Val Loss   : 0.0057 | Val Acc   : 0.9975
  Overfit Gap (val - train loss): -0.0248

EPOCH [9/10]
  Epoch [9/10] | Batch [10/102] | Loss: 0.0045 | Acc: 0.9969
  Epoch [9/10] | Batch [20/102] | Loss: 0.0037 | Acc: 0.9969
  Epoch [9/10] | Batch [30/102] | Loss: 0.0024 | Acc: 0.9948
  Epoch [9/10] | Batch [40/102] | Loss: 0.0020 | Acc: 0.9938
  Epoch [9/10] | Batch [50/102] | Loss: 0.0005 | Acc: 0.9950
  Epoch [9/10] | Batch [60/102] | Loss: 0.0062 | Acc: 0.9938
  Epoch [9/10] | Batch [70/102] | Loss: 0.0101 | Acc: 0.9929
  Epoch [9/10] | Batch [80/102] | Loss: 0.0264 | Acc: 0.9934
  Epoch [9/10] | Batch [90/102] | Loss: 0.0107 | Acc: 0.9927
  Epoch [9/10] | Batch [100/102] | Loss: 0.0

We observe overfitting  in multiple hyperparameter combinations. This possibly due to our small dataset. Further regularization and counter overfitting techniques will be tested.